# Final Semester Project: Deep Vision Pipeline

**Student IDs:** 231192, 231200  
**Course:** AI303 Computer Vision  
**Dataset:** Brain Tumor MRI slices, continuing Assignments 1-3

This Colab notebook completes the final project requirement by building an end-to-end pipeline:

1. Acquisition: download or upload the MRI dataset.
2. Pre-processing: radiometric normalization and denoising from Assignment 1.
3. Segmentation: use annotated tumor masks when available, otherwise use the Assignment 2 style classical mask fallback.
4. Description: compute GLCM and geometric descriptors from Assignment 3.
5. Recognition: train a deep model and evaluate it with confusion matrix plus precision/recall analysis.
6. Export: save the annotated dataset, trained model, metrics, plots, report draft, and GitHub-ready files as a zip.

Run the notebook from top to bottom in Google Colab. At the end, it downloads one zip file. Send that zip back so the final report and GitHub packaging can be finished with the real results.

## 1. Install Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("nibabel", "nibabel"),
    ("scikit-image", "skimage"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("kagglehub", "kagglehub"),
    ("tqdm", "tqdm"),
    ("scikit-learn", "sklearn"),
    ("pandas", "pandas"),
    ("seaborn", "seaborn"),
    ("pillow", "PIL"),
    ("python-docx", "docx"),
]

missing = [pip_name for pip_name, import_name in REQUIRED
           if importlib.util.find_spec(import_name) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All non-TensorFlow dependencies are already available.")

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
except Exception:
    print("TensorFlow is not available. Installing tensorflow. This can take a few minutes.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)

## 2. Imports and Configuration

In [ ]:
import csv
import gc
import json
import os
import random
import re
import shutil
import textwrap
import warnings
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image
from scipy import ndimage as ndi
from skimage.color import gray2rgb
from skimage.exposure import rescale_intensity
from skimage.feature import canny, graycomatrix, graycoprops
from skimage.filters import gaussian, threshold_otsu
from skimage.measure import label as sk_label
from skimage.measure import regionprops
from skimage.morphology import binary_closing, binary_opening, disk, remove_small_objects
from skimage.transform import resize
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Running in Colab:", IN_COLAB)
print("TensorFlow devices:", tf.config.list_physical_devices())

In [ ]:
# Edit these values only if you need a faster or larger run.
PROJECT_NAME = "231192_231200_Final_Deep_Vision_Pipeline"
DATASET_SLUG = "awansaad6797/cancer-dataset"

# Set USE_UPLOAD=True if Kaggle download fails and you want to upload a zip manually.
USE_UPLOAD = False
LOCAL_DATASET_DIR = None

# Deep model choice:
# "3d_transformer" satisfies the research challenge by learning over neighboring MRI slices.
# "2d_cnn" is faster and can be used as a fallback on slower runtimes.
MODEL_KIND = "3d_transformer"

IMAGE_SIZE = 96
CONTEXT_DEPTH = 5
MAX_SUBJECTS = 18
MAX_VOLUMES = 48
POSITIVE_SLICES_PER_VOLUME = 40
NEGATIVE_SLICES_PER_VOLUME = 40
CENTER_SLICES_WITHOUT_MASK = 64

MIN_TUMOR_PIXELS = 15
MIN_FALLBACK_MASK_PIXELS = 40

EPOCHS = 20
BATCH_SIZE = 8 if MODEL_KIND == "3d_transformer" else 32
TEST_SIZE = 0.20
VAL_SIZE_FROM_TRAIN = 0.15

BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()
OUT_DIR = BASE_DIR / "final_deep_vision_outputs"
DATASET_OUT = OUT_DIR / "annotated_dataset"
IMAGE_OUT = DATASET_OUT / "images"
MASK_OUT = DATASET_OUT / "masks"
OVERLAY_OUT = DATASET_OUT / "overlays"
FIG_OUT = OUT_DIR / "figures"
MODEL_OUT = OUT_DIR / "models"
FEATURE_OUT = OUT_DIR / "features"
REPORT_OUT = OUT_DIR / "report"
GITHUB_OUT = OUT_DIR / "github_ready"

for folder in [IMAGE_OUT, MASK_OUT, OVERLAY_OUT, FIG_OUT, MODEL_OUT, FEATURE_OUT, REPORT_OUT, GITHUB_OUT]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)
print("Model kind:", MODEL_KIND)

## 3. Acquire the Dataset

In [ ]:
def acquire_dataset():
    if USE_UPLOAD:
        upload_dir = BASE_DIR / "uploaded_dataset"
        upload_dir.mkdir(parents=True, exist_ok=True)
        if not IN_COLAB:
            if LOCAL_DATASET_DIR is None:
                raise ValueError("Set LOCAL_DATASET_DIR when USE_UPLOAD=True outside Colab.")
            return Path(LOCAL_DATASET_DIR)
        print("Upload a dataset zip file containing .nii or .nii.gz files.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        first_file = next(iter(uploaded))
        archive_path = upload_dir / first_file
        shutil.unpack_archive(str(archive_path), str(upload_dir))
        return upload_dir

    print("Downloading dataset through kagglehub...")
    path = kagglehub.dataset_download(DATASET_SLUG)
    return Path(path)


DATA_ROOT = acquire_dataset()
print("Dataset root:", DATA_ROOT)
print("NIfTI files found:", len(list(DATA_ROOT.rglob("*.nii"))) + len(list(DATA_ROOT.rglob("*.nii.gz"))))

## 4. Discover Volumes and Pair Tumor Masks

In [ ]:
def strip_nii_suffix(path):
    name = Path(path).name
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return Path(path).stem


def is_segmentation_name(name):
    low = strip_nii_suffix(str(name)).lower()
    tokens = ["seg", "mask", "label", "labels", "annotation"]
    return any(re.search(rf"(^|[-_]){token}($|[-_])", low) for token in tokens)


def modality_from_name(name):
    low = name.lower()
    if is_segmentation_name(low):
        return "seg"
    if any(token in low for token in ["t1ce", "t1c", "t1gd", "ce"]):
        return "t1ce"
    if "t1n" in low or re.search(r"(^|[-_])t1($|[-_])", low):
        return "t1"
    if "t2f" in low or "flair" in low or re.search(r"(^|[-_])fl($|[-_])", low):
        return "flair"
    if "t2w" in low or re.search(r"(^|[-_])t2($|[-_])", low):
        return "t2"
    return "unknown"


def subject_key(name):
    stem = strip_nii_suffix(name).lower()
    stem = re.sub(r"(_to_sri_defaced|_defaced)$", "", stem)
    stem = re.sub(r"[-_](t1ce|t1c|t1gd|t1n|t1|t2f|flair|fl|t2w|t2|seg|mask|label|labels|annotation)$", "", stem)
    stem = re.sub(r"[^a-z0-9]+", "_", stem).strip("_")
    return stem or strip_nii_suffix(name).lower()


def safe_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")


all_nii = sorted(set(DATA_ROOT.rglob("*.nii")) | set(DATA_ROOT.rglob("*.nii.gz")))
records = []
for path in all_nii:
    stem = strip_nii_suffix(path)
    records.append({
        "path": path,
        "name": path.name,
        "stem": stem,
        "subject": subject_key(path.name),
        "modality": modality_from_name(path.name),
        "is_seg": is_segmentation_name(path.name),
    })

volume_df = pd.DataFrame(records)
display(volume_df.head(20))

seg_files = volume_df[volume_df["is_seg"]].copy()
image_files = volume_df[~volume_df["is_seg"]].copy()

segs_by_subject = {}
for _, row in seg_files.iterrows():
    segs_by_subject.setdefault(row["subject"], []).append(Path(row["path"]))

image_groups = {}
for _, row in image_files.iterrows():
    image_groups.setdefault(row["subject"], []).append(row.to_dict())

segmented_subjects = sorted(set(image_groups) & set(segs_by_subject))
if segmented_subjects:
    selected_subjects = segmented_subjects[:MAX_SUBJECTS]
    recognition_task = "tumor_presence"
else:
    selected_subjects = sorted(image_groups)[:MAX_SUBJECTS]
    recognition_task = "modality"

selected_images = []
for subject in selected_subjects:
    selected_images.extend(image_groups.get(subject, []))
selected_images = selected_images[:MAX_VOLUMES]

print("Total image volumes:", len(image_files))
print("Total segmentation volumes:", len(seg_files))
print("Subjects with masks:", len(segmented_subjects))
print("Selected subjects:", len(selected_subjects))
print("Selected image volumes:", len(selected_images))
print("Recognition task:", recognition_task)

## 5. Pipeline Helper Functions

In [ ]:
def load_nifti(path):
    img = nib.load(str(path))
    vol = img.get_fdata(dtype=np.float32)
    vol = np.squeeze(vol)
    if vol.ndim == 4:
        vol = vol[..., 0]
    if vol.ndim != 3:
        raise ValueError(f"Expected 3D NIfTI volume, got shape {vol.shape}: {path}")
    return np.nan_to_num(vol.astype(np.float32))


def normalise_slice(slc):
    slc = np.nan_to_num(slc.astype(np.float32))
    nonzero = slc[slc > 0]
    sample = nonzero if nonzero.size > 10 else slc.ravel()
    lo, hi = np.percentile(sample, [1, 99])
    if hi <= lo:
        lo, hi = float(slc.min()), float(slc.max())
    if hi <= lo:
        return np.zeros_like(slc, dtype=np.float32)
    return np.clip((slc - lo) / (hi - lo + 1e-7), 0, 1).astype(np.float32)


def preprocess_slice(slc):
    slc = normalise_slice(slc)
    slc = gaussian(slc, sigma=0.65, preserve_range=True)
    slc = ndi.median_filter(slc, size=3)
    slc = ndi.uniform_filter(slc, size=2)
    return np.clip(slc, 0, 1).astype(np.float32)


def resize_image(slc, size=IMAGE_SIZE):
    return resize(slc, (size, size), order=1, mode="reflect",
                  anti_aliasing=True, preserve_range=True).astype(np.float32)


def resize_mask(mask, size=IMAGE_SIZE):
    return (resize(mask.astype(np.float32), (size, size), order=0, mode="edge",
                   anti_aliasing=False, preserve_range=True) > 0.5).astype(np.uint8)


def fallback_mask_from_slice(slc):
    smoothed = gaussian(slc, sigma=1.0, preserve_range=True)
    foreground = smoothed > 0.05
    if foreground.sum() < MIN_FALLBACK_MASK_PIXELS:
        return np.zeros_like(slc, dtype=np.uint8)
    try:
        thresh = threshold_otsu(smoothed[foreground])
        coarse = smoothed > thresh
    except Exception:
        coarse = foreground
    edges = canny(smoothed, sigma=1.25)
    mask = np.logical_or(coarse, edges)
    mask = binary_closing(mask, footprint=disk(3))
    mask = binary_opening(mask, footprint=disk(2))
    mask = ndi.binary_fill_holes(mask)
    mask = remove_small_objects(mask.astype(bool), min_size=MIN_FALLBACK_MASK_PIXELS)
    labeled = sk_label(mask)
    props = regionprops(labeled)
    if not props:
        return np.zeros_like(slc, dtype=np.uint8)
    largest = max(props, key=lambda p: p.area)
    return (labeled == largest.label).astype(np.uint8)


def segmentation_mask(seg_vol, z):
    if seg_vol is None:
        return None
    z = int(np.clip(z, 0, seg_vol.shape[2] - 1))
    return (seg_vol[:, :, z] > 0).astype(np.uint8)


def choose_evenly(indices, limit):
    indices = list(map(int, indices))
    if len(indices) <= limit:
        return indices
    picks = np.linspace(0, len(indices) - 1, limit).round().astype(int)
    return [indices[i] for i in picks]


def select_slice_indices(depth, seg_vol=None):
    if seg_vol is not None and seg_vol.shape[2] == depth:
        counts = np.array([(seg_vol[:, :, z] > 0).sum() for z in range(depth)])
        pos = np.where(counts >= MIN_TUMOR_PIXELS)[0]
        neg = np.where(counts < MIN_TUMOR_PIXELS)[0]
        pos_sorted = pos[np.argsort(counts[pos])[::-1]] if len(pos) else np.array([], dtype=int)
        pos_pick = sorted(pos_sorted[:POSITIVE_SLICES_PER_VOLUME].tolist())
        neg_pick = choose_evenly(neg, NEGATIVE_SLICES_PER_VOLUME)
        combined = sorted(set(pos_pick + neg_pick))
        if combined:
            return combined

    center = depth // 2
    half = CENTER_SLICES_WITHOUT_MASK // 2
    start = max(0, center - half)
    end = min(depth, start + CENTER_SLICES_WITHOUT_MASK)
    return list(range(start, end))


def context_from_volume(vol, z):
    half = CONTEXT_DEPTH // 2
    slices = []
    for dz in range(-half, half + 1):
        zi = int(np.clip(z + dz, 0, vol.shape[2] - 1))
        slices.append(resize_image(preprocess_slice(vol[:, :, zi])))
    context = np.stack(slices, axis=0)[..., np.newaxis]
    return context.astype(np.float32)


def save_png(path, array):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    img = np.clip(array, 0, 1)
    Image.fromarray((img * 255).astype(np.uint8)).save(path)


def save_mask_png(path, mask):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)


def save_overlay(path, image, mask):
    rgb = gray2rgb(np.clip(image, 0, 1))
    overlay = rgb.copy()
    overlay[mask.astype(bool)] = [1.0, 0.05, 0.05]
    blended = 0.65 * rgb + 0.35 * overlay
    Image.fromarray((np.clip(blended, 0, 1) * 255).astype(np.uint8)).save(path)


def compute_descriptor_features(image, mask):
    active = mask.astype(bool)
    region = image.copy()
    if active.any():
        region[~active] = 0
    img_uint = np.clip(region * 63, 0, 63).astype(np.uint8)
    glcm = graycomatrix(
        img_uint,
        distances=[1, 2],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=64,
        symmetric=True,
        normed=True,
    )
    glcm_plane = glcm[:, :, 0, 0]
    entropy = float(-np.sum(glcm_plane * np.log2(glcm_plane + 1e-10)))
    labeled = sk_label(mask)
    props = regionprops(labeled)
    if props:
        obj = max(props, key=lambda p: p.area)
        area = float(obj.area)
        perimeter = float(obj.perimeter)
        centroid_r, centroid_c = map(float, obj.centroid)
        circularity = float((4 * np.pi * area) / (perimeter ** 2 + 1e-8))
    else:
        area = perimeter = centroid_r = centroid_c = circularity = 0.0

    return {
        "energy": float(graycoprops(glcm, "energy").mean()),
        "entropy": entropy,
        "contrast": float(graycoprops(glcm, "contrast").mean()),
        "homogeneity": float(graycoprops(glcm, "homogeneity").mean()),
        "correlation": float(graycoprops(glcm, "correlation").mean()),
        "area": area,
        "perimeter": perimeter,
        "centroid_r": centroid_r,
        "centroid_c": centroid_c,
        "circularity": circularity,
    }

## 6. Build the Annotated Dataset Used for Training and Testing

In [ ]:
def label_from_modality(modality):
    mapping = {
        "t1": "t1_native",
        "t1ce": "t1_contrast",
        "t2": "t2_weighted",
        "flair": "flair",
    }
    return mapping.get(modality, "unknown_modality")


samples = []
manifest_rows = []

for rec in tqdm(selected_images, desc="Volumes"):
    img_path = Path(rec["path"])
    subject = rec["subject"]
    modality = rec["modality"]
    seg_path = segs_by_subject.get(subject, [None])[0]

    try:
        vol = load_nifti(img_path)
        seg_vol = None
        if seg_path is not None:
            seg_candidate = load_nifti(seg_path)
            if seg_candidate.shape == vol.shape:
                seg_vol = seg_candidate
            else:
                print(f"Mask shape mismatch for {img_path.name}: image={vol.shape}, mask={seg_candidate.shape}")

        for z in select_slice_indices(vol.shape[2], seg_vol):
            context = context_from_volume(vol, z)
            center_image = context[CONTEXT_DEPTH // 2, :, :, 0]
            if center_image.max() < 0.01:
                continue

            raw_mask = segmentation_mask(seg_vol, z)
            if raw_mask is not None:
                mask = resize_mask(raw_mask)
                label = "tumor" if mask.sum() >= MIN_TUMOR_PIXELS else "no_tumor"
                label_source = "segmentation"
            else:
                mask = fallback_mask_from_slice(center_image)
                label = label_from_modality(modality)
                label_source = "filename_modality"

            if label == "unknown_modality":
                continue

            sample_id = f"{safe_name(subject)}_{safe_name(modality)}_z{int(z):04d}_{len(samples):05d}"
            image_path = IMAGE_OUT / f"{sample_id}.png"
            mask_path = MASK_OUT / f"{sample_id}_mask.png"
            overlay_path = OVERLAY_OUT / f"{sample_id}_overlay.png"

            save_png(image_path, center_image)
            save_mask_png(mask_path, mask)
            save_overlay(overlay_path, center_image, mask)

            feats = compute_descriptor_features(center_image, mask)
            row = {
                "sample_id": sample_id,
                "subject": subject,
                "source_volume": img_path.name,
                "segmentation_volume": Path(seg_path).name if seg_path is not None else "",
                "modality": modality,
                "slice_index": int(z),
                "label": label,
                "label_source": label_source,
                "image_path": str(image_path.relative_to(OUT_DIR)),
                "mask_path": str(mask_path.relative_to(OUT_DIR)),
                "overlay_path": str(overlay_path.relative_to(OUT_DIR)),
                **feats,
            }
            manifest_rows.append(row)
            samples.append({
                "id": sample_id,
                "x3d": context,
                "x2d": center_image[..., np.newaxis].astype(np.float32),
                "label": label,
            })

        del vol
        gc.collect()
    except Exception as exc:
        print(f"Skipping {img_path.name}: {exc}")

manifest_df = pd.DataFrame(manifest_rows)
if manifest_df.empty:
    raise RuntimeError("No training samples were created. Check the dataset path and NIfTI files.")

label_counts = manifest_df["label"].value_counts()
display(label_counts.to_frame("count"))

if manifest_df["label"].nunique() < 2:
    raise RuntimeError(
        "Only one class was found. Increase MAX_SUBJECTS/MAX_VOLUMES or verify that segmentation masks are present."
    )

features_csv = FEATURE_OUT / "deep_pipeline_features.csv"
manifest_csv = DATASET_OUT / "annotated_dataset_manifest.csv"
manifest_df.to_csv(manifest_csv, index=False)
manifest_df.to_csv(features_csv, index=False)

print("Samples:", len(samples))
print("Labels:", label_counts.to_dict())
print("Manifest:", manifest_csv)
print("Feature CSV:", features_csv)

## 7. Visualize Pre-processing, Segmentation, and Descriptor Features

In [ ]:
preview_n = min(8, len(manifest_df))
preview_rows = manifest_df.sample(preview_n, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(preview_n, 3, figsize=(9, 2.5 * preview_n))
if preview_n == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, row in preview_rows.iterrows():
    image = np.array(Image.open(OUT_DIR / row["image_path"])) / 255.0
    mask = np.array(Image.open(OUT_DIR / row["mask_path"])) > 0
    overlay = np.array(Image.open(OUT_DIR / row["overlay_path"])) / 255.0

    axes[row_idx, 0].imshow(image, cmap="gray")
    axes[row_idx, 0].set_title(f"{row['label']} | z={row['slice_index']}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(mask, cmap="gray")
    axes[row_idx, 1].set_title("Mask")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(overlay)
    axes[row_idx, 2].set_title("Overlay")
    axes[row_idx, 2].axis("off")

fig.tight_layout()
preview_path = FIG_OUT / "preprocessing_segmentation_preview.png"
fig.savefig(preview_path, dpi=140, bbox_inches="tight")
plt.show()
print("Saved:", preview_path)

descriptor_cols = ["energy", "entropy", "contrast", "homogeneity", "correlation", "area", "perimeter", "circularity"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), descriptor_cols):
    sns.boxplot(data=manifest_df, x="label", y=col, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=25)
fig.suptitle("Assignment 3 Descriptors Reused in the Final Pipeline", y=1.02)
fig.tight_layout()
descriptor_path = FIG_OUT / "descriptor_distributions.png"
fig.savefig(descriptor_path, dpi=140, bbox_inches="tight")
plt.show()
print("Saved:", descriptor_path)

## 8. Prepare Deep Learning Splits

In [ ]:
labels = np.array([sample["label"] for sample in samples])
encoder = LabelEncoder()
y_all = encoder.fit_transform(labels)
class_names = encoder.classes_.tolist()
num_classes = len(class_names)

all_indices = np.arange(len(samples))
min_class_count = min(np.bincount(y_all))
stratify_labels = y_all if min_class_count >= 2 else None

train_val_idx, test_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=stratify_labels,
)

y_train_val = y_all[train_val_idx]
stratify_train_val = y_train_val if min(np.bincount(y_train_val)) >= 2 else None
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=VAL_SIZE_FROM_TRAIN,
    random_state=SEED,
    stratify=stratify_train_val,
)

def make_x(indices):
    if MODEL_KIND == "3d_transformer":
        return np.stack([samples[int(i)]["x3d"] for i in indices]).astype(np.float32)
    return np.stack([samples[int(i)]["x2d"] for i in indices]).astype(np.float32)


X_train = make_x(train_idx)
X_val = make_x(val_idx)
X_test = make_x(test_idx)
y_train = y_all[train_idx].astype(np.int32)
y_val = y_all[val_idx].astype(np.int32)
y_test = y_all[test_idx].astype(np.int32)

manifest_df["split"] = ""
manifest_df.loc[train_idx, "split"] = "train"
manifest_df.loc[val_idx, "split"] = "validation"
manifest_df.loc[test_idx, "split"] = "test"
manifest_df.to_csv(DATASET_OUT / "annotated_dataset_manifest.csv", index=False)

print("Classes:", class_names)
print("Train:", X_train.shape, np.bincount(y_train, minlength=num_classes))
print("Validation:", X_val.shape, np.bincount(y_val, minlength=num_classes))
print("Test:", X_test.shape, np.bincount(y_test, minlength=num_classes))

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1024, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

present_classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=present_classes, y=y_train)
class_weight = {int(cls): float(weight) for cls, weight in zip(present_classes, weights)}
print("Class weights:", class_weight)

## 9. Define the Deep Model

In [ ]:
from tensorflow.keras import layers


def output_layer(x, num_classes):
    if num_classes == 2:
        return layers.Dense(1, activation="sigmoid", name="tumor_probability")(x)
    return layers.Dense(num_classes, activation="softmax", name="class_probability")(x)


def compile_model(model, num_classes):
    if num_classes == 2:
        loss = "binary_crossentropy"
        metrics = [
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ]
    else:
        loss = "sparse_categorical_crossentropy"
        metrics = ["accuracy"]
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=loss,
        metrics=metrics,
    )
    return model


def build_2d_cnn(num_classes):
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1), name="preprocessed_mri_slice")
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.15)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.35)(x)
    outputs = output_layer(x, num_classes)
    return compile_model(tf.keras.Model(inputs, outputs, name="A1_A2_A3_2D_CNN"), num_classes)


def transformer_encoder(tokens, token_dim, heads=2, dropout=0.10):
    attn = layers.MultiHeadAttention(num_heads=heads, key_dim=max(8, token_dim // heads), dropout=dropout)(tokens, tokens)
    x = layers.Add()([tokens, attn])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    ff = layers.Dense(token_dim * 2, activation="gelu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(token_dim)(ff)
    x = layers.Add()([x, ff])
    return layers.LayerNormalization(epsilon=1e-6)(x)


def build_3d_transformer(num_classes):
    inputs = tf.keras.Input(shape=(CONTEXT_DEPTH, IMAGE_SIZE, IMAGE_SIZE, 1), name="mri_slice_context")

    x = layers.Conv3D(16, kernel_size=(3, 5, 5), strides=(1, 2, 2), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2))(x)

    x = layers.Conv3D(32, kernel_size=3, strides=(1, 2, 2), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.15)(x)

    token_dim = int(x.shape[-1])
    x = layers.Reshape((-1, token_dim), name="voxel_patch_tokens")(x)
    x = transformer_encoder(x, token_dim=token_dim, heads=2, dropout=0.10)
    x = transformer_encoder(x, token_dim=token_dim, heads=2, dropout=0.10)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(96, activation="relu")(x)
    x = layers.Dropout(0.35)(x)
    outputs = output_layer(x, num_classes)
    return compile_model(tf.keras.Model(inputs, outputs, name="A1_A2_A3_3D_Transformer"), num_classes)


if MODEL_KIND == "3d_transformer":
    model = build_3d_transformer(num_classes)
else:
    model = build_2d_cnn(num_classes)

model.summary()

## 10. Train the Model

In [ ]:
best_model_path = MODEL_OUT / f"{PROJECT_NAME}_{MODEL_KIND}.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(str(best_model_path), monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)

history_df = pd.DataFrame(history.history)
history_csv = OUT_DIR / "training_history.csv"
history_df.to_csv(history_csv, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["loss"], label="train")
axes[0].plot(history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

if "accuracy" in history_df:
    axes[1].plot(history_df["accuracy"], label="train")
    axes[1].plot(history_df["val_accuracy"], label="validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

fig.tight_layout()
curves_path = FIG_OUT / "training_curves.png"
fig.savefig(curves_path, dpi=140, bbox_inches="tight")
plt.show()
print("Best model:", best_model_path)
print("Training history:", history_csv)

## 11. Evaluate with Confusion Matrix and Precision/Recall

In [ ]:
if best_model_path.exists():
    model = tf.keras.models.load_model(best_model_path)

decision_threshold = 0.5
if num_classes == 2:
    val_scores = model.predict(val_ds).reshape(-1)
    val_precision, val_recall, val_thresholds = precision_recall_curve(y_val, val_scores)
    if len(val_thresholds):
        val_f1 = 2 * val_precision[:-1] * val_recall[:-1] / (val_precision[:-1] + val_recall[:-1] + 1e-8)
        decision_threshold = float(val_thresholds[int(np.nanargmax(val_f1))])
    print(f"Validation-tuned decision threshold: {decision_threshold:.4f}")

raw_pred = model.predict(test_ds)
if num_classes == 2:
    probabilities = raw_pred.reshape(-1)
    pred_ids = (probabilities >= decision_threshold).astype(int)
    score_for_positive = probabilities
else:
    probabilities = raw_pred
    pred_ids = raw_pred.argmax(axis=1)
    score_for_positive = raw_pred.max(axis=1)

true_names = encoder.inverse_transform(y_test)
pred_names = encoder.inverse_transform(pred_ids)

report_dict = classification_report(
    y_test,
    pred_ids,
    labels=np.arange(num_classes),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_text = classification_report(
    y_test,
    pred_ids,
    labels=np.arange(num_classes),
    target_names=class_names,
    zero_division=0,
)
print(report_text)

report_csv = OUT_DIR / "classification_report.csv"
pd.DataFrame(report_dict).transpose().to_csv(report_csv)

cm = confusion_matrix(y_test, pred_ids, labels=np.arange(num_classes))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix - {MODEL_KIND}")
fig.tight_layout()
cm_path = FIG_OUT / "confusion_matrix.png"
fig.savefig(cm_path, dpi=160, bbox_inches="tight")
plt.show()

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_test, pred_ids, labels=np.arange(num_classes), average="macro", zero_division=0
)

metrics_summary = {
    "project_name": PROJECT_NAME,
    "model_kind": MODEL_KIND,
    "recognition_task": recognition_task,
    "num_samples": int(len(samples)),
    "num_classes": int(num_classes),
    "class_names": class_names,
    "test_accuracy": float((pred_ids == y_test).mean()),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "confusion_matrix": cm.tolist(),
    "decision_threshold": float(decision_threshold),
}

if num_classes == 2:
    precision, recall, thresholds = precision_recall_curve(y_test, score_for_positive)
    pr_auc = auc(recall[::-1], precision[::-1])
    metrics_summary["precision_recall_auc"] = float(pr_auc)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall, precision, label=f"PR AUC={pr_auc:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve")
    ax.legend()
    fig.tight_layout()
    pr_path = FIG_OUT / "precision_recall_curve.png"
    fig.savefig(pr_path, dpi=160, bbox_inches="tight")
    plt.show()

metrics_json = OUT_DIR / "metrics_summary.json"
with open(metrics_json, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

prediction_df = pd.DataFrame({
    "sample_index": test_idx,
    "sample_id": [samples[int(i)]["id"] for i in test_idx],
    "true_label": true_names,
    "predicted_label": pred_names,
    "score": score_for_positive,
})
prediction_csv = OUT_DIR / "test_predictions.csv"
prediction_df.to_csv(prediction_csv, index=False)

manifest_df["predicted_label"] = ""
manifest_df["prediction_score"] = ""
for _, row in prediction_df.iterrows():
    idx = int(row["sample_index"])
    manifest_df.loc[idx, "predicted_label"] = row["predicted_label"]
    manifest_df.loc[idx, "prediction_score"] = row["score"]
manifest_df.to_csv(DATASET_OUT / "annotated_dataset_manifest.csv", index=False)

print("Classification report:", report_csv)
print("Metrics summary:", metrics_json)
print("Predictions:", prediction_csv)

## 12. Save Sample Predictions

In [ ]:
plot_rows = prediction_df.sample(min(12, len(prediction_df)), random_state=SEED).reset_index(drop=True)
cols = 4
rows = int(np.ceil(len(plot_rows) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
axes = np.array(axes).reshape(-1)

for ax in axes:
    ax.axis("off")

for ax, (_, row) in zip(axes, plot_rows.iterrows()):
    sample = samples[int(row["sample_index"])]
    image = sample["x2d"][:, :, 0]
    ok = row["true_label"] == row["predicted_label"]
    ax.imshow(image, cmap="gray")
    ax.set_title(
        f"T: {row['true_label']}\nP: {row['predicted_label']} ({row['score']:.2f})",
        color="green" if ok else "red",
        fontsize=9,
    )
    ax.axis("off")

fig.suptitle("Held-out Test Predictions", y=1.02)
fig.tight_layout()
pred_grid_path = FIG_OUT / "sample_predictions.png"
fig.savefig(pred_grid_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", pred_grid_path)

## 13. Generate Report Draft and GitHub-Ready Files

In [ ]:
requirements_text = "\n".join([
    "nibabel",
    "scikit-image",
    "scipy",
    "matplotlib",
    "kagglehub",
    "tqdm",
    "scikit-learn",
    "pandas",
    "seaborn",
    "pillow",
    "tensorflow",
    "python-docx",
]) + "\n"
(GITHUB_OUT / "requirements.txt").write_text(requirements_text, encoding="utf-8")

readme_text = f"""# {PROJECT_NAME}

Final Computer Vision project continuing Assignments 1-3.

## Pipeline

1. Acquisition: KaggleHub dataset `{DATASET_SLUG}` or uploaded NIfTI data.
2. Pre-processing: percentile radiometric normalization plus Gaussian, median, and mean denoising.
3. Segmentation: paired tumor masks when present, with a classical fallback mask for unannotated data.
4. Description: GLCM texture features and geometric features.
5. Recognition: `{MODEL_KIND}` deep model.
6. Evaluation: confusion matrix, precision/recall, classification report, and held-out predictions.

## Current Run

- Recognition task: {recognition_task}
- Samples: {len(samples)}
- Classes: {', '.join(class_names)}
- Test accuracy: {metrics_summary['test_accuracy']:.4f}
- Macro precision: {metrics_summary['macro_precision']:.4f}
- Macro recall: {metrics_summary['macro_recall']:.4f}
- Macro F1: {metrics_summary['macro_f1']:.4f}

## Main Artifacts

- `annotated_dataset/`: image slices, masks, overlays, and manifest.
- `features/deep_pipeline_features.csv`: Assignment 3 descriptors reused in the final pipeline.
- `models/`: trained Keras model.
- `figures/`: confusion matrix, training curves, descriptor plots, and prediction examples.
- `classification_report.csv` and `metrics_summary.json`: final evaluation.
"""
(GITHUB_OUT / "README.md").write_text(readme_text, encoding="utf-8")

report_md = f"""# Final Semester Project Report Draft

## Title
Deep Vision Pipeline for Brain Tumor MRI Slice Understanding

## Abstract
This project extends the first three Computer Vision assignments into an end-to-end deep vision workflow. The system performs data acquisition, radiometric pre-processing, segmentation, feature description, and final recognition using a deep model. The current run used `{MODEL_KIND}` for the recognition stage and produced a held-out test accuracy of {metrics_summary['test_accuracy']:.4f}, macro precision of {metrics_summary['macro_precision']:.4f}, macro recall of {metrics_summary['macro_recall']:.4f}, and macro F1 of {metrics_summary['macro_f1']:.4f}.

## Literature Review Notes
Medical image analysis pipelines commonly combine radiometric correction, denoising, segmentation, handcrafted descriptors, and deep learning. GLCM features capture local texture statistics, while geometric descriptors capture object shape and compactness. CNNs learn hierarchical image representations directly from pixels. For the bonus research challenge, the 3D transformer option uses neighboring MRI slices so attention can model voxel-to-voxel dependencies across slice context.

## Methodology
The dataset was acquired from KaggleHub using `{DATASET_SLUG}`. Each NIfTI volume was normalized with percentile clipping, denoised with Gaussian, median, and mean filtering, and converted to 2D slice samples with a local 3D context window. If a paired tumor segmentation volume was available, it was used as the annotation source. Otherwise, a classical segmentation fallback based on thresholding, Canny edges, morphology, hole filling, and connected components was used for descriptor extraction.

## Feature Description
For every sample, the pipeline exported GLCM energy, entropy, contrast, homogeneity, and correlation. It also exported area, perimeter, centroid, and circularity for the segmented region.

## Deep Recognition Model
The selected model was `{MODEL_KIND}`. The default 3D transformer first embeds neighboring MRI slices with Conv3D layers, flattens the result into voxel-patch tokens, and applies multi-head self-attention. This connects the final project to the 3D transformer research challenge.

## Results
- Samples: {len(samples)}
- Recognition task: {recognition_task}
- Classes: {', '.join(class_names)}
- Test accuracy: {metrics_summary['test_accuracy']:.4f}
- Macro precision: {metrics_summary['macro_precision']:.4f}
- Macro recall: {metrics_summary['macro_recall']:.4f}
- Macro F1: {metrics_summary['macro_f1']:.4f}

The confusion matrix, descriptor distributions, training curves, and prediction examples are saved in the `figures/` directory.

## Conclusion
The final pipeline integrates the pre-processing, structural segmentation, texture/geometric description, and classification components developed during Assignments 1-3. The exported annotated dataset and metrics can be used for the final 15-page paper and GitHub submission.
"""
report_md_path = REPORT_OUT / "Final_Project_Report_Draft.md"
report_md_path.write_text(report_md, encoding="utf-8")

try:
    from docx import Document
    doc = Document()
    doc.add_heading("Deep Vision Pipeline for Brain Tumor MRI Slice Understanding", 0)
    doc.add_paragraph("Student IDs: 231192, 231200")
    for section_title in [
        "Abstract",
        "Literature Review Notes",
        "Methodology",
        "Feature Description",
        "Deep Recognition Model",
        "Results",
        "Conclusion",
    ]:
        marker = f"## {section_title}"
        if marker in report_md:
            content = report_md.split(marker, 1)[1].split("\n## ", 1)[0].strip()
            doc.add_heading(section_title, level=1)
            for paragraph in content.split("\n\n"):
                doc.add_paragraph(paragraph.strip())
    docx_path = REPORT_OUT / "Final_Project_Report_Draft.docx"
    doc.save(docx_path)
    print("DOCX report draft:", docx_path)
except Exception as exc:
    print("DOCX report draft skipped:", exc)

print("Markdown report draft:", report_md_path)
print("GitHub-ready README:", GITHUB_OUT / "README.md")
print("GitHub-ready requirements:", GITHUB_OUT / "requirements.txt")

## 14. Zip and Download All Results

In [ ]:
zip_base = BASE_DIR / PROJECT_NAME
zip_path = shutil.make_archive(str(zip_base), "zip", OUT_DIR)
print("Created zip:", zip_path)

if IN_COLAB:
    files.download(zip_path)
else:
    print("Not running in Colab, so the zip was not auto-downloaded.")